In [7]:
# Install needed libraries
%pip install -U python-jobspy
%pip install tqdm
%pip install xlsxwriter
# Install MongoDB Python driver
%pip install pymongo
%pip install python-dotenv

You should consider upgrading via the '/home/wagner/.pyenv/versions/3.10.0/envs/market_scrapper_venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
You should consider upgrading via the '/home/wagner/.pyenv/versions/3.10.0/envs/market_scrapper_venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
You should consider upgrading via the '/home/wagner/.pyenv/versions/3.10.0/envs/market_scrapper_venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
You should consider upgrading via the '/home/wagner/.pyenv/versions/3.10.0/envs/market_scrapper_venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
You should consider upgrading via the '/home/wagner/.pyenv/versions/3.10.0/envs/market_scrapper_venv/bin/python -m pip install --upg

In [ ]:
import sys
from pathlib import Path

# Get the absolute path of the project root (one level up from the notebooks directory)
project_root = str(Path().resolve().parent)  # Goes up two levels to reach the project root

# Add the project root to the Python path
if project_root not in sys.path:
    sys.path.append(project_root)

# Now import your modules
from operations import process_and_save_jobs, setup_output_directory, connect_to_mongodb

ModuleNotFoundError: No module named 'operations'

In [ ]:
from jobspy import scrape_jobs
from operations import process_and_save_jobs, setup_output_directory, connect_to_mongodb
import itertools

ModuleNotFoundError: No module named 'operations'

In [ ]:
connect_to_mongodb()

Looking for .env at: /home/wagner/Documentos/dev-projects/No Country/Market-Scraper/.env
✅ Successfully connected to MongoDB
📊 Database: job_market
📂 Collection: jobs
🔗 Total documents: 10553


{'client': MongoClient(host=['ac-fky0ob9-shard-00-02.ncfzs7b.mongodb.net:27017', 'ac-fky0ob9-shard-00-01.ncfzs7b.mongodb.net:27017', 'ac-fky0ob9-shard-00-00.ncfzs7b.mongodb.net:27017'], document_class=dict, tz_aware=False, connect=True, appname='Cluster0', authsource='admin', replicaset='atlas-t6534q-shard-0', tls=True, serverselectiontimeoutms=5000),
 'collection': Collection(Database(MongoClient(host=['ac-fky0ob9-shard-00-02.ncfzs7b.mongodb.net:27017', 'ac-fky0ob9-shard-00-01.ncfzs7b.mongodb.net:27017', 'ac-fky0ob9-shard-00-00.ncfzs7b.mongodb.net:27017'], document_class=dict, tz_aware=False, connect=True, appname='Cluster0', authsource='admin', replicaset='atlas-t6534q-shard-0', tls=True, serverselectiontimeoutms=5000), 'job_market'), 'jobs')}

In [ ]:
# --- 1. Definir Directorio de Salida ---
output_dir = setup_output_directory("../data/raw")
print(f"Directorio de salida: {output_dir}")

Directorio de salida: ../data/raw/jobs_20251127_175346


In [ ]:
# --- 2. Definir Parámetros de Búsqueda Base ---
sectores_clave = ["Fintech", "EdTech", "Future of Work"]
search_terms = sectores_clave
location = "Remote"

"""indeed_glassdoor_countries = [
    "Australia",
    "Austria",
    "Belgium",
    "Brazil",
    "Canada",
    "France",
    "Germany",
    "Hong Kong",
    "India",
    "Ireland",
    "Italy",
    "Mexico",
    "Netherlands",
    "New Zealand",
    "Singapore",
    "Spain",
    "Switzerland",
    "UK",
    "USA",
    "Vietnam"
]"""

indeed_glassdoor_countries = [
    "Australia",
    "Austria"
]
country_code = "USA" # Código de país para Indeed/Glassdoor

In [ ]:
# --- 3. Lista para guardar resultados ---
# Guardaremos los DataFrames de cada sitio aquí
all_jobs_dfs = []

print("Parámetros listos. Iniciaremos scrapers secuenciales y especializados.")

Parámetros listos. Iniciaremos scrapers secuenciales y especializados.


In [ ]:
# --- 1. Scraper: Indeed (El "Caballo de batalla") ---
# Es el más estable y sin límites de solicitudes

print("\n--- Iniciando Scraper: Indeed/Glassdoor ---")
for country_indeed, search_term in itertools.product(indeed_glassdoor_countries, search_terms):
    print(f"Buscando en {country_indeed} por {search_term}")
    try:
        indeed_jobs = scrape_jobs(
            site_name=["indeed","glassdoor"],
            search_term=search_term, # Podemos afinar el término
            location=location,
            country_indeed=country_indeed,
            results_wanted=99999, # Le pedimos más porque es estable
            hours_old=720
        )
        if indeed_jobs is not None and not indeed_jobs.empty:
                print(f"✅ Indeed encontró {len(indeed_jobs)} trabajos.")
                all_jobs_dfs.append(indeed_jobs)
    except Exception as e:
        print(f"❌ Error en Indeed: {e}")


--- Iniciando Scraper: Indeed/Glassdoor ---
Buscando en Australia por Fintech
✅ Indeed encontró 4 trabajos.
Buscando en Australia por EdTech
✅ Indeed encontró 4 trabajos.
Buscando en Australia por Future of Work
✅ Indeed encontró 4 trabajos.
Buscando en Austria por Fintech
✅ Indeed encontró 2 trabajos.
Buscando en Austria por EdTech
Buscando en Austria por Future of Work
✅ Indeed encontró 4 trabajos.


In [ ]:
"""
# --- 3. Scraper: ZipRecruiter (El "Estándar") ---
print("\n--- Iniciando Scraper: ZipRecruiter ---")
for item in search_terms:
    search_term = f'"{item[0]}" AND "{item[1]}"'
    print(f"Buscando: {search_term}")
    try:
        zip_jobs = scrape_jobs(
            site_name=["zip_recruiter"],
            search_term=search_term,
            location=location,
            results_wanted=100, 
            hours_old=720
        )
        if zip_jobs is not None and not zip_jobs.empty:
            print(f"✅ ZipRecruiter encontró {len(zip_jobs)} trabajos.")
            all_jobs_dfs.append(zip_jobs)
    except Exception as e:
        print(f"❌ Error en ZipRecruiter: {e}")
"""


'\n# --- 3. Scraper: ZipRecruiter (El "Estándar") ---\nprint("\n--- Iniciando Scraper: ZipRecruiter ---")\nfor item in search_terms:\n    search_term = f\'"{item[0]}" AND "{item[1]}"\'\n    print(f"Buscando: {search_term}")\n    try:\n        zip_jobs = scrape_jobs(\n            site_name=["zip_recruiter"],\n            search_term=search_term,\n            location=location,\n            results_wanted=100, \n            hours_old=720\n        )\n        if zip_jobs is not None and not zip_jobs.empty:\n            print(f"✅ ZipRecruiter encontró {len(zip_jobs)} trabajos.")\n            all_jobs_dfs.append(zip_jobs)\n    except Exception as e:\n        print(f"❌ Error en ZipRecruiter: {e}")\n'

In [ ]:
"""print("\n--- Iniciando Scraper: Glassdoor ---")
for item in search_terms:
    search_term = f'"{item[0]}" AND "{item[1]}"'
    print(f"Buscando: {search_term}")
    try:
        zip_jobs = scrape_jobs(
            site_name=["glassdoor"],
            search_term=search_term,
            location=location,
            results_wanted=100, 
            hours_old=720
        )
        if zip_jobs is not None and not zip_jobs.empty:
            print(f"✅ GlassDoor encontró {len(zip_jobs)} trabajos.")
            all_jobs_dfs.append(zip_jobs)
    except Exception as e:
        print(f"❌ Error en Glassdoor: {e}")"""

'print("\n--- Iniciando Scraper: Glassdoor ---")\nfor item in search_terms:\n    search_term = f\'"{item[0]}" AND "{item[1]}"\'\n    print(f"Buscando: {search_term}")\n    try:\n        zip_jobs = scrape_jobs(\n            site_name=["glassdoor"],\n            search_term=search_term,\n            location=location,\n            results_wanted=100, \n            hours_old=720\n        )\n        if zip_jobs is not None and not zip_jobs.empty:\n            print(f"✅ GlassDoor encontró {len(zip_jobs)} trabajos.")\n            all_jobs_dfs.append(zip_jobs)\n    except Exception as e:\n        print(f"❌ Error en Glassdoor: {e}")'

In [ ]:
# --- 2. Scraper: LinkedIn (El "Delicado") ---
# Alto riesgo de 429. Lo llamamos con cuidado.
"""print("\n--- Iniciando Scraper: LinkedIn ---")
for item in search_terms:
    search_term = f'"{item[0]}" AND "{item[1]}"'
    print(f"Buscando: {search_term}")
    try:
        linkedin_jobs = scrape_jobs(
            site_name=["linkedin"],
            search_term=search_term,
            location=location,
            results_wanted=50, # MUY BAJO para evitar 429 sin proxies
            hours_old=720,
            linkedin_fetch_description=True # Clave para enriquecimiento
        )
        if linkedin_jobs is not None and not linkedin_jobs.empty:
            print(f"✅ LinkedIn encontró {len(linkedin_jobs)} trabajos.")
            all_jobs_dfs.append(linkedin_jobs)
    except Exception as e:
        print(f"❌ Error en LinkedIn: {e}. (Probablemente Error 429)")
"""

'print("\n--- Iniciando Scraper: LinkedIn ---")\nfor item in search_terms:\n    search_term = f\'"{item[0]}" AND "{item[1]}"\'\n    print(f"Buscando: {search_term}")\n    try:\n        linkedin_jobs = scrape_jobs(\n            site_name=["linkedin"],\n            search_term=search_term,\n            location=location,\n            results_wanted=50, # MUY BAJO para evitar 429 sin proxies\n            hours_old=720,\n            linkedin_fetch_description=True # Clave para enriquecimiento\n        )\n        if linkedin_jobs is not None and not linkedin_jobs.empty:\n            print(f"✅ LinkedIn encontró {len(linkedin_jobs)} trabajos.")\n            all_jobs_dfs.append(linkedin_jobs)\n    except Exception as e:\n        print(f"❌ Error en LinkedIn: {e}. (Probablemente Error 429)")\n'

In [ ]:
print("\n--- Scraping secuencial completado ---")


--- Scraping secuencial completado ---


In [ ]:
# Update your main processing loop:
if all_jobs_dfs:
    process_and_save_jobs(all_jobs_dfs, output_dir)
else:
    print("\nNo jobs were found.")

/home/wagner/Documentos/dev-projects/No Country/Market-Scraper/operations/process_job_data.py:36: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(job_dfs, ignore_index=True).drop_duplicates(


Looking for .env at: /home/wagner/Documentos/dev-projects/No Country/Market-Scraper/.env
✅ Successfully connected to MongoDB
📊 Database: job_market
📂 Collection: jobs
🔗 Total documents: 10557

📊 Job Processing Summary
✅ New jobs inserted: 0
🔄 Existing jobs updated: 0
⏩ Jobs unchanged (skipped): 18
❌ Errors: 0
📦 Backup saved to: ../data/raw/jobs_20251127_175346/backup_20251127_175437.json
